In [1]:
import numpy as np 
from scipy.optimize import minimize 
import xarray as xr 
import xmitgcm
import matplotlib.pyplot as plt
from scipy.optimize import basinhopping
import itertools
import optuna
from scipy.io import loadmat
import gsw
from cmcrameri import cm


/rds/general/user/sl8924/home/anaconda3/envs/stats_env/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
#This code runs the model simulations 

def Euler_step(T_1, T_2, S_1, S_2, t, dt, params):
    T_1_star, S_1_star, T_2_star, S_2_star, tau_1_T, tau_1_S, tau_2, A_T, A_S, psi=params
    #First integration
    dT1dt = (1/tau_1_T) * (T_1_star - A_T*np.cos(2*np.pi*t) -T_1)
    dS1dt = (1/tau_1_S) * (S_1_star + A_S*np.cos(2*np.pi*t + psi) - S_1)
    dT2dt = (1/tau_2) * (T_2_star - T_2)
    dS2dt = (1/tau_2) * (S_2_star - S_2)

    T_1_new = T_1 + dT1dt * dt 
    S_1_new = S_1 + dS1dt * dt 
    T_2_new = T_2 + dT2dt * dt 
    S_2_new = S_2 + dS2dt *dt 

    #Convective adjustment if statically unstable 
    
    Delta_rho = -alpha * (T_1-T_2) + beta * (S_1 - S_2)
    if Delta_rho > 0 :
        T_1_new = h_star*T_1_new + (1-h_star)*T_2_new 
        T_2_new = T_1_new 
        S_1_new = h_star*S_1_new + (1-h_star)*S_2_new
        S_2_new = S_1_new 

    else:
        T_1_new, T_2_new, S_1_new, S_2_new = T_1_new, T_2_new, S_1_new, S_2_new
 
    return T_1_new, T_2_new, S_1_new, S_2_new 

#Run the time integration 
def run_euler(T_1, T_2, S_1, S_2, T_max, dt, params):
    T_1_list = []
    T_2_list = []
    S_1_list = []
    S_2_list = [] 
    tlist = []
    t = 0
    #Run simulation 
    while t <= T_max:
        T_1, T_2, S_1, S_2 = Euler_step(T_1, T_2, S_1, S_2, t, dt, params)
        tlist.append(t)
        T_1_list.append(T_1)
        T_2_list.append(T_2)
        S_1_list.append(S_1)
        S_2_list.append(S_2)
        t += dt 
    
    return tlist, T_1_list, T_2_list, S_1_list, S_2_list 

def simulate_model(T_max, dt, params):
    tlist, T_1sim, T_2sim, S_1sim, S_2sim = run_euler(T_1init, T_2init, S_1init, S_2init, T_max, dt, params)
    return tlist[0::30], T_1sim[0::30], T_2sim[0::30], S_1sim[0::30], S_2sim[0::30]

In [3]:
#ASTE data 
file_path = #Insert filepath
T_1_data200 = xr.open_dataset(file_path)
file_path = #Insert filepath
S_1_data200 = xr.open_dataset(file_path)
file_path = #Insert filepath
T_2_data200 = xr.open_dataset(file_path)
file_path = #Insert filepath
S_2_data200 = xr.open_dataset(file_path)

T_1obs200 = T_1_data200.THETA.values[:-1]
S_1obs200 = S_1_data200.SALT.values[:-1]
T_2obs200 = T_2_data200.THETA.values[:-1]
S_2obs200 = S_2_data200.SALT.values[:-1]

T_1init200, T_2init200, S_1init200, S_2init200 = T_1obs200[0], T_2obs200[0], S_1obs200[0], S_2obs200[0]

file_path = #Insert filepath
T_1_data50 = xr.open_dataset(file_path)
file_path = #Insert filepath
S_1_data50 = xr.open_dataset(file_path)
file_path = #Insert filepath
T_2_data50 = xr.open_dataset(file_path)
file_path = #Insert filepath
S_2_data50 = xr.open_dataset(file_path)

T_1obs50 = T_1_data50.THETA.values[:-1]
S_1obs50 = S_1_data50.SALT.values[:-1]
T_2obs50 = T_2_data50.THETA.values[:-1]
S_2obs50 = S_2_data50.SALT.values[:-1]

T_1init50, T_2init50, S_1init50, S_2init50 = T_1obs50[0], T_2obs50[0], S_1obs50[0], S_2obs50[0]

#Expansion coefficients 
alpha = 1
beta = 10

#Parameter sets 
params50m = [3.99, 34.16, 4.04, 34.89, 0.23, 7.47, 1.78, 5.97, 3.97, 0.1]
params200m= [3.84, 34.62, 4, 34.88, 0.32, 6.43, 1.41, 2.78, 2.99, 0.14]


SyntaxError: invalid syntax (2012057912.py, line 2)

In [ ]:
#Create Figure 3 
params = params50m
dt = 1/360
alpha = 1
beta = 10
h_star = 1/36
#This demonstrates the nonconvective attractor 
T_1init, T_2init, S_1init, S_2init = T_1init50, T_2init50, S_1init50, S_2init50
x = simulate_model(16, dt, params)
#This demonstrates the convective attractor
T_1init, T_2init, S_1init, S_2init = 3.9, 3.9, 34.9, 34.9
y = simulate_model(16, dt, params)

#Plot the ASTE data
fig, ax = plt.subplots(4, 1, figsize=(6,9))
ax[0].plot(x[0], x[1], label='Upper box', color='blue')
ax[0].plot(x[0], x[2], label='Lower box', color='red')
ax[0].set_ylabel('Temperature ($^\circ$C)')

ax[1].set_ylabel('Salinity (psu)')
ax[1].plot(x[0], x[3], color='blue')
ax[1].plot(x[0], x[4], color='red')

ax[2].plot(y[0], y[1], color='blue')
ax[2].plot(y[0], y[2], color='red')
ax[2].set_ylabel('Temperature ($^\circ$C)')


ax[3].set_ylabel('Salinity (psu)')
ax[3].plot(y[0], y[3], color='blue')
ax[3].plot(y[0], y[4], color='red')
ax[3].set_xlabel('Time (years)')
plt.tight_layout()
ax[0].legend()
plt.show()


In [ ]:
#This code produces Figure 4
params = params50m
dt = 1/360
alpha = 1
beta = 10
h_star = 1/36
y = simulate_model(16, dt, params)
time = [i+2002 for i in y[0]]

#Plot the ASTE data
fig, ax = plt.subplots(2, 1, figsize=(8,7))
ax[1].plot(time, S_1obs50, label='Upper box data', color='blue')
ax[1].plot(time, S_2obs50, label='Lower box data', color='red')
ax[1].set_ylabel('Salinity (psu)')


ax[0].set_ylabel('Temperature ($^\circ$C)')
ax[0].plot(time, T_1obs50, label='Upper box', color='blue')
ax[0].plot(time, T_2obs50, label='Lower box', color='red')
ax[1].set_xlabel('Year')
plt.legend()
plt.show()

In [ ]:
#This code uses optuna to optimise parameters for the model 
#For 50m box
h_star = 1/36
#Defining cost function (50m box)
def cost_function50(params):
    tlist, T_1sim, T_2sim, S_1sim, S_2sim = simulate_model(16, dt, params)
     # Compute squared error
    T1anom = T_1sim - T_1obs50 
    T2anom = T_2sim - T_2obs50 
    S1anom = S_1sim - S_1obs50
    S2anom = S_2sim - S_2obs50
    err_T1 = 1/192 * np.sum((T1anom)**2)
    err_T2 = 1/192 * np.sum((T2anom)**2)
    err_S1 = 1/192 * np.sum((S1anom)**2)
    err_S2 = 1/192 * np.sum((S2anom)**2)
    
    return err_T1 + err_T2 + err_S1 + err_S2


#Optuna optimisation (50m box)
def objective50(trial):
    T_1_star = trial.suggest_float("T_1_star", 2.5, 4.5) 
    S_1_star = trial.suggest_float("S_1_star", 33.8, 34.8)      
    T_2_star = trial.suggest_float("T_2_star", 3.8, 4.6) 
    S_2_star = trial.suggest_float("S_2_star", 34.86, 34.9)
    tau_1_T = trial.suggest_float("tau_1_T", 0.2, 0.7)
    tau_1_S = trial.suggest_float("tau_1_S", 3, 10)
    tau_2 = trial.suggest_float("tau_2", 1, 3)
    A_T = trial.suggest_float("A_T", 3, 7)
    A_S = trial.suggest_float("A_S", 2, 5)
    psi = trial.suggest_float("psi", 0, 0.3)
    
    # Evaluate cost function
    loss = cost_function50((T_1_star, S_1_star, T_2_star, S_2_star, tau_1_T, tau_1_S, tau_2, A_T, A_S, psi))

    # Optional: pruning if your function supports intermediate reporting
    trial.report(loss, step=1)
    if trial.should_prune():
        raise optuna.TrialPruned()

    return loss
    # Report intermediate results (for pruning)
    trial.report(score, step=1)

    if trial.should_prune():
        raise optuna.TrialPruned()

    return score
optuna.logging.set_verbosity(optuna.logging.WARNING)
# Run study
study = optuna.create_study(direction="minimize")
study.optimize(objective50, n_trials=1000)

print("Best score:", study.best_trial.value)
print("Best params:", study.best_trial.params)


In [ ]:
#This code uses optuna to optimise parameters for the model 
#For 200m box
h_star = 1/10
#Defining cost function (200m box)
def cost_function200(params):
    tlist, T_1sim, T_2sim, S_1sim, S_2sim = simulate_model(16, dt, params)
     # Compute squared error
    T1anom = T_1sim - T_1obs200 
    T2anom = T_2sim - T_2obs200 
    S1anom = S_1sim - S_1obs200
    S2anom = S_2sim - S_2obs200
    err_T1 = 1/192 * np.sum((T1anom)**2)
    err_T2 = 1/192 * np.sum((T2anom)**2)
    err_S1 = 1/192 * np.sum((S1anom)**2)
    err_S2 = 1/192 * np.sum((S2anom)**2)
    
    return err_T1 + err_T2 + err_S1 + err_S2

#Optuna optimisation (200m box)
def objective200(trial):
    T_1_star = trial.suggest_float("T_1_star", 2.5, 4.5) 
    S_1_star = trial.suggest_float("S_1_star", 33.8, 34.8)      
    T_2_star = trial.suggest_float("T_2_star", 3.8, 4.6) 
    S_2_star = trial.suggest_float("S_2_star", 34.8, 34.9)
    tau_1_T = trial.suggest_float("tau_1_T", 0.2, 0.7)
    tau_1_S = trial.suggest_float("tau_1_S", 3, 10)
    tau_2 = trial.suggest_float("tau_2", 1, 4)
    A_T = trial.suggest_float("A_T", 2, 4)
    A_S = trial.suggest_float("A_S", 2, 4)
    psi = trial.suggest_float("psi", 0, 0.3)
    
    # Evaluate cost function
    loss = cost_function200((T_1_star, S_1_star, T_2_star, S_2_star, tau_1_T, tau_1_S, tau_2, A_T, A_S, psi))

    # Optional: pruning if your function supports intermediate reporting
    trial.report(loss, step=1)
    if trial.should_prune():
        raise optuna.TrialPruned()

    return loss
    # Report intermediate results (for pruning)
    trial.report(score, step=1)

    if trial.should_prune():
        raise optuna.TrialPruned()

    return score
optuna.logging.set_verbosity(optuna.logging.WARNING)
# Run study
study = optuna.create_study(direction="minimize")
study.optimize(objective200, n_trials=1000)

print("Best score:", study.best_trial.value)
print("Best params:", study.best_trial.params)

In [ ]:
#Create a list of 10 optimised parameters; choose either 50m or 200m box
params_list_list = []
optuna.logging.set_verbosity(optuna.logging.WARNING)
for j in range(10):
    studyj = optuna.create_study(direction="minimize")
    studyj.optimize(objective, n_trials=1000)
    best_params_dictj = studyj.best_trial.params
    best_params_listj = list(best_params_dictj.values())
    params_list_list.append(best_params_listj)

params_table = np.zeros((10, 10))
for i in range(10):
    for j in range(10):
        params_table[i][j] += params_list_list[j][i]

In [ ]:
#This code produces Figure 5

#Running simulation using optimal parameters
T_1init, T_2init, S_1init, S_2init = T_1init50, T_2init50, S_1init50, S_2init50
params = params50m
dt = 1/360
alpha = 1
beta = 10
h_star = 1/36
y = simulate_model(16, dt, params)

fig, ax = plt.subplots(2, 1, figsize=(8,7))
ax[1].plot(y[0], S_1obs50, label='Upper box data', color='blue')
ax[1].plot(y[0], y[3], color='green', label='Upper box model fit')
ax[1].plot(y[0], S_2obs50, label='Lower box data', color='red')
ax[1].plot(y[0], y[4], color='orange', label='Lower box model fit')
ax[1].set_ylabel('Salinity (psu)')


ax[0].set_ylabel('Temperature ($^\circ$C)')
ax[0].plot(y[0], T_1obs50, label='Upper box', color='blue')
ax[0].plot(y[0], y[1], color='green')
ax[0].plot(y[0], T_2obs50, label='Lower box', color='red')
ax[0].plot(y[0], y[2], color='orange')
ax[1].set_xlabel('Year')
plt.legend()
plt.show()

In [ ]:
#Creating Figure 6 of the Argo data 
#Load data 
file_path = #Insert filepath
argo = loadmat(file_path)
argo['salt'].shape

#Putting the Argo data into a useable form 
time = (argo['time'])/365.24 # Better units of time 
salt = argo['salt']
salt = gsw.SP_from_SA(salt, 1000*9.81*argo['depth'], 307.35, 56.58)
temp = argo['temp']
print(argo['time'][48]-argo['time'][0]) #This is exactly two years after the start of the dataset. 
                                        #If we run from here we have more or less 21 years of data to use.
                                        #Datapoints are 15.2 days apart. We have 504 datapoints.
salt_useable = salt[:, 48:541] #Since the final 11 datapoints are all nans as well, we omit them too. 
print(len(salt_useable[100])) #In total we have 493 datapoints spaced at 15.2 days apart.
                                #If a year is 365 days, we have 20.5 years

salt0_to_200m = salt_useable[0:20, :] # First 200m 
salt0_to_200m_mean = np.mean(salt0_to_200m, axis=0)
salt200_to_2000m = salt_useable[20:185, :] #200m to 1850m since limited data at bottom layers mess things up 
salt200_to_2000m_mean = np.mean(salt200_to_2000m, axis=0)
fig, ax = plt.subplots(2, 1, figsize=(8,7))
ax[1].plot(time[48:541], salt0_to_200m_mean, label='Upper box', color='blue')
ax[1].plot(time[48:541], salt200_to_2000m_mean, label='Lower box', color='red')
ax[1].set_ylabel('Salinity (psu)')

temp_useable = temp[:, 48:541]
temp0_to_200m = temp_useable[0:20, :] # First 200m 
temp0_to_200m_mean = np.mean(temp0_to_200m, axis=0)
temp200_to_2000m = temp_useable[20:185, :] #200m to 1850m since limited data at bottom layers mess things up 
temp200_to_2000m_mean = np.mean(temp200_to_2000m, axis=0)
ax[0].set_ylabel('Temperature ($^\circ$C)')
ax[0].plot(time[48:541], temp0_to_200m_mean, label='Upper box', color='blue')
ax[0].plot(time[48:541], temp200_to_2000m_mean, label='Lower box', color='red')
ax[1].set_xlabel('Year')
plt.legend()
plt.show()

In [ ]:
#Produce Figure 7, which tests the condition for the reachability of the 'off' state

#Small timestep for accuracy 
dt = 1/3600
#Start with convection off
T_1init = 3.55
T_2init = 3.61
S_1init = 34.3
S_2init = 34.87
alpha = 1
beta = 10
h_star = 1/36
#Two parameter sets; for one, convection should remain off, for the othger, convection should switch back on
params1 = [3.99, 34.468, 4.04, 34.89, 0.23, 7.47, 1.78, 5.97, 3.97, 0.1]
params2 = [3.99, 34.469, 4.04, 34.89, 0.23, 7.47, 1.78, 5.97, 3.97, 0.1]

#Test condition 24
T_1_star, S_1_star, T_2_star, S_2_star, tau_1_T, tau_1_S, tau_2, A_T, A_S, psi = params1
#Calculate relevant quantities
lhs = (beta/alpha) * (S_1_star - S_2_star) - (T_1_star - T_2_star)
print(lhs)
Q_T = A_T/np.sqrt(1+ (2*np.pi * tau_1_T)**2)
Q_S = (beta/alpha) * (A_S/np.sqrt(1 + (2*np.pi * tau_1_S)**2))
phi_T = np.pi - np.arctan(2*np.pi*tau_1_T)
phi_S = psi - np.arctan(2*np.pi*tau_1_S)
R = np.sqrt(Q_T**2 + Q_S**2 - 2*Q_T*Q_S*np.cos(phi_T - phi_S))
print("-R = "+str(-R))
lhs = (beta/alpha) * (S_1_star - S_2_star) - (T_1_star - T_2_star)
print("lhs = "+str(lhs))
#If the following is negative, convection should remain off. If positive, convection should restart
print(R + lhs)

#For second parameter set 
T_1_star, S_1_star, T_2_star, S_2_star, tau_1_T, tau_1_S, tau_2, A_T, A_S, psi = params2
#Calculate relevant quantities 
lhs = (beta/alpha) * (S_1_star - S_2_star) - (T_1_star - T_2_star)
Q_T = A_T/np.sqrt(1+ (2*np.pi * tau_1_T)**2)
Q_S = (beta/alpha) * (A_S/np.sqrt(1 + (2*np.pi * tau_1_S)**2))
phi_T = np.pi - np.arctan(2*np.pi*tau_1_T)
phi_S = psi - np.arctan(2*np.pi*tau_1_S)
R = np.sqrt(Q_T**2 + Q_S**2 - 2*Q_T*Q_S*np.cos(phi_T - phi_S))
print("-R = "+str(-R))
lhs = (beta/alpha) * (S_1_star - S_2_star) - (T_1_star - T_2_star)
print("lhs = "+str(lhs))
#If the following is negative, convection should remain off. If positive, convection should restart
print(R + lhs)

#Run model for 200 years to test for restarting of convection, and plot 
y = simulate_model(200, dt, params1)
x = simulate_model(200, dt, params2)
fig, axs = plt.subplots(2, 1, sharex=True, sharey=False, figsize=(7,8))

axs[0].plot(y[0], y[3], label='Upper box')
axs[0].plot(y[0], y[4], label='Lower box')
axs[0].set_ylabel("Salinity (psu)")
axs[0].set_title('a)', loc='left')

axs[1].plot(x[0], x[3])
axs[1].plot(x[0], x[4])
axs[1].set_ylabel("Salinity (psu)")
axs[1].set_title('b)', loc='left')
axs[1].set_xlabel('Time (years)')
fig.legend()


In [ ]:
#Produce Figure 8 
h_star = 1/36
dt = 1/10000
#We want convection to start being on 
T_1init = 3.55
T_2init = 3.61
S_1init = 39.78
S_2init = 34.87
#Two different parameter sets. For one, the column should remain permanently with convection active. For the other,
#convection should periodically cease.
params1 = [3.99, 57.5, 4.04, 34.89, 0.23, 7.47, 1.78, 5.97, 3.97, 0.1]
params2 = [3.99, 58, 4.04, 34.89, 0.23, 7.47, 1.78, 5.97, 3.97, 0.1]
y = simulate_model(16, dt, params1)
x = simulate_model(16, dt, params2)
t = y[0]

# Convection on solution for parameter set 1
kS1 = ((params1[1] * h_star * params1[6]) + params1[3] * params1[5]) / (params1[5] + h_star * params1[6])
absBS1 = params1[8] / (np.sqrt((1 + (params1[5] / (h_star * params1[6]))**2 + (2 * np.pi * params1[5] * ((h_star + 1) / h_star))**2)))
argBS1 = np.arctan(params1[5] * 2 * np.pi * ((h_star + 1) / h_star) / (1 + params1[5] / (h_star * params1[6])))
Sconvect1 = [kS1 + absBS1 * np.cos(2 * np.pi * q + params1[9] - argBS1) for q in t]  # S solution

kT1 = ((params1[0] * h_star * params1[6]) + (params1[2] * params1[4])) / (params1[4] + h_star * params1[6])
absB1 = params1[7] / (np.sqrt((1 + (params1[4] / (h_star * params1[6]))**2 + (2 * np.pi * params1[4] * ((h_star + 1) / h_star))**2)))
argB1 = np.arctan(params1[4] * 2 * np.pi * ((h_star + 1) / h_star) / (1 + params1[4] / (h_star * params1[6])))
Tconvect1 = [kT1 + absB1 * np.cos(2 * np.pi * q + np.pi - argB1) for q in t]  # T solution

Scosterm1 = [ (params1[8]/params1[5])*np.cos(2*np.pi*q  + params1[9]) for q in t]
Tcosterm1 = [(params1[7]/params1[4])*np.cos(2*np.pi*q + np.pi) for q in t]

salt_term1 = [beta*(-((params1[1]/params1[5])-(params1[3]/params1[6]) + Sconvect1[q]*(1/params1[6] - 1/params1[5]) ) + Scosterm1[q]) for q in range(len(t))]
temp_term1 = [alpha*(-((params1[0]/params1[4])-(params1[2]/params1[6]) + Tconvect1[q]*(1/params1[6] - 1/params1[4])) + Tcosterm1[q]) for q in range(len(t))]

# Convection on solution for parameter set 2
kS2 = ((params2[1] * h_star * params2[6]) + params2[3] * params2[5]) / (params2[5] + h_star * params2[6])
absBS2 = params2[8] / (np.sqrt((1 + (params2[5] / (h_star * params2[6]))**2 + (2 * np.pi * params2[5] * ((h_star + 1) / h_star))**2)))
argBS2 = np.arctan(params2[5] * 2 * np.pi * ((h_star + 1) / h_star) / (1 + params2[5] / (h_star * params2[6])))
Sconvect2 = [kS2 + absBS2 * np.cos(2 * np.pi * q + params2[9] - argBS2) for q in t]  # S solution

kT2 = ((params2[0] * h_star * params2[6]) + (params2[2] * params2[4])) / (params2[4] + h_star * params2[6])
absB2 = params2[7] / (np.sqrt((1 + (params2[4] / (h_star * params2[6]))**2 + (2 * np.pi * params2[4] * ((h_star + 1) / h_star))**2)))
argB2 = np.arctan(params2[4] * 2 * np.pi * ((h_star + 1) / h_star) / (1 + params2[4] / (h_star * params2[6])))
Tconvect2 = [kT2 + absB2 * np.cos(2 * np.pi * q + np.pi - argB2) for q in t]  # T solution

Scosterm2 = [(params2[8] / params2[5]) * np.cos(2 * np.pi * q + params2[9]) for q in t]
Tcosterm2 = [(params2[7] / params2[4]) * np.cos(2 * np.pi * q + np.pi) for q in t]

#The following are the key terms in (28)
salt_term2 = [beta * (-((params2[1] / params2[5]) - (params2[3] / params2[6]) + Sconvect2[q] * (1 / params2[6] - 1 / params2[5])) + Scosterm2[q]) for q in range(len(t))]
temp_term2 = [alpha * (-((params2[0] / params2[4]) - (params2[2] / params2[6]) + Tconvect2[q] * (1 / params2[6] - 1 / params2[4])) + Tcosterm2[q]) for q in range(len(t))]

#Plot 
fig, ax = plt.subplots(4, 1, figsize=(7,10), sharex=True, sharey=False)
ax[0].plot(y[0], y[3], label='Upper box')
ax[0].plot(y[0], y[4], label='Lower box')
ax[0].set_ylabel('Salinity (psu)')
ax[0].set_title('a)', loc='left')
ax[0].legend()
ax[1].plot(y[0], salt_term1, label='Salinity curve', color='blue')
ax[1].plot(y[0], temp_term1, label='Temperature curve', color='red')
ax[1].set_title('b)', loc='left')
ax[1].legend()
ax[2].plot(x[0], x[3])
ax[2].plot(x[0], x[4])
ax[2].set_ylabel('Salinity (psu)')
ax[2].set_title('c)', loc='left')
ax[3].plot(y[0], salt_term2, color='blue')
ax[3].plot(y[0], temp_term2, color='red')
ax[3].set_xlabel('Time (years)')
ax[3].set_title('d)', loc='left')
fig.savefig('testing29.png')
fig.show()

#If positive, convection should periodically cease. If negative, convection should remain permanently active 
print(max(salt_term1) - min(temp_term1))
print(max(salt_term2) - min(temp_term2))

In [ ]:
#Creating Figure 9; calculate bifurcation geometry (takes a while)
#This is for the 50m box 

params50m = [3.99, 34.16, 4.04, 34.89, 0.23, 7.47, 1.78, 5.97, 3.97, 0.1]
params=params50m
h_star = 1/36
T_1init, T_2init, S_1init, S_2init = T_1init50, T_2init50, S_1init50, S_2init50
S1range = [33 + 0.05*i for i in range(40)]
T1range = [3 + 0.05*i for i in range(40)]
T_grid, S_grid = np.meshgrid(T1range, S1range)
indicator = []

#Test convective breakdown 
for t, s in zip(T_grid.ravel(), S_grid.ravel()):
    params = [t,s]
    T2 = simulate_model(200, dt, [params[0], params[1],  4.04, 34.89, 0.23, 7.47, 1.78, 5.97, 3.97, 0.1 ])[2]
    if np.isclose(T2[-1] - T2[-2], 0, atol=1e-5) == True:
        indicator.append(1)
    else:
        indicator.append(0) 
cmap = plt.get_cmap('coolwarm', 2)  
Z = np.array(indicator).reshape(T_grid.shape)
fig = plt.figure(figsize=(6,5))
p = plt.pcolor(T_grid, S_grid, Z, cmap='coolwarm')
plt.xlabel("$T_1^*$")
plt.ylabel("$S_1^*$")
params = params50m


In [ ]:
#Create final figure
#Include the boundary to the 'On' region
T_1_star, S_1_star, T_2_star, S_2_star, tau_1_T, tau_1_S, tau_2, A_T, A_S, psi = params
lhs = (beta/alpha) * (S_1_star - S_2_star) - (T_1_star - T_2_star)
Q_T = A_T/np.sqrt(1+ (2*np.pi * tau_1_T)**2)
Q_S = (beta/alpha) * (A_S/np.sqrt(1 + (2*np.pi * tau_1_S)**2))
phi_T = np.pi - np.arctan(2*np.pi*tau_1_T)
phi_S = psi - np.arctan(2*np.pi*tau_1_S)
R = np.sqrt(Q_T**2 + Q_S**2 - 2*Q_T*Q_S*np.cos(phi_T - phi_S))


T1list = [2.99 + 0.01*i for i in range(199)]
LHS = np.array([-(alpha/beta) * R + (alpha/beta) * (k - T_2_star) + S_2_star for k in T1list])
kuhlbrodt = [alpha/beta * (k - A_T - T_2_star) - A_S + S_2_star for k in T1list]
cmap = plt.get_cmap('coolwarm', 2)
cmap = plt.get_cmap('coolwarm', 2)
kuhlbrodt2 = [(alpha/beta)*((tau_1_S+h_star*tau_2)/(tau_1_T+h_star*tau_2))*(k-A_T-T_2_star)-A_S+S_2_star for k in T1list]


fig = plt.figure(figsize=(6,5))
p = plt.pcolor(T_grid, S_grid, Z, cmap=cm.batlow)
plt.xlabel("$T_1^*$")
plt.ylabel("$S_1^*$")

plt.plot(T1list, LHS, color='crimson')
plt.fill_between(T1list, LHS, 34.97, color='crimson', alpha=1)
plt.scatter(3.99, 34.16, marker='*', s=100, color='yellow')

plt.show()


In [ ]:
#This is the 200m box element of Figure 9; calculate bifurcation geometry (takes a while)

params=params200m
h_star = 1/10
T_1init, T_2init, S_1init, S_2init = T_1init200, T_2init200, S_1init200, S_2init200
S1range = [33 + 0.05*i for i in range(40)]
T1range = [3 + 0.05*i for i in range(40)]
T_grid, S_grid = np.meshgrid(T1range, S1range)
indicator = []

#Test convective breakdown 
for t, s in zip(T_grid.ravel(), S_grid.ravel()):
    params = [t,s]
    T2 = simulate_model(200, dt, [params[0], params[1], 4, 34.88, 0.32, 6.43, 1.41, 2.78, 2.99, 0.14])[2]
    if np.isclose(T2[-1] - T2[-2], 0, atol=1e-5) == True:
        indicator.append(1)
    else:
        indicator.append(0) 
cmap = plt.get_cmap('coolwarm', 2)  
Z = np.array(indicator).reshape(T_grid.shape)
fig = plt.figure(figsize=(6,5))
p = plt.pcolor(T_grid, S_grid, Z, cmap='coolwarm')
plt.xlabel("$T_1^*$")
plt.ylabel("$S_1^*$")
params = params200m



In [ ]:
#Include the boundary to the 'On' region
T_1_star, S_1_star, T_2_star, S_2_star, tau_1_T, tau_1_S, tau_2, A_T, A_S, psi = params
lhs = (beta/alpha) * (S_1_star - S_2_star) - (T_1_star - T_2_star)
Q_T = A_T/np.sqrt(1+ (2*np.pi * tau_1_T)**2)
Q_S = (beta/alpha) * (A_S/np.sqrt(1 + (2*np.pi * tau_1_S)**2))
phi_T = np.pi - np.arctan(2*np.pi*tau_1_T)
phi_S = psi - np.arctan(2*np.pi*tau_1_S)
R = np.sqrt(Q_T**2 + Q_S**2 - 2*Q_T*Q_S*np.cos(phi_T - phi_S))
print("-R = "+str(-R))
lhs = (beta/alpha) * (S_1_star - S_2_star) - (T_1_star - T_2_star)
print("lhs = "+str(lhs))

T1list = [2.99 + 0.01*i for i in range(199)]
LHS = np.array([-(alpha/beta) * R + (alpha/beta) * (k - T_2_star) + S_2_star for k in T1list])
kuhlbrodt = [alpha/beta * (k - A_T - T_2_star) - A_S + S_2_star for k in T1list]
cmap = plt.get_cmap('coolwarm', 2)
cmap = plt.get_cmap('coolwarm', 2)
kuhlbrodt2 = [(alpha/beta)*((tau_1_S+h_star*tau_2)/(tau_1_T+h_star*tau_2))*(k-A_T-T_2_star)-A_S+S_2_star for k in T1list]


fig = plt.figure(figsize=(6,5))
p = plt.pcolor(T_grid, S_grid, Z, cmap=cm.batlow)
plt.xlabel("$T_1^*$")
plt.ylabel("$S_1^*$")

plt.plot(T1list, LHS, color='crimson')
plt.fill_between(T1list, LHS, 34.97, color='crimson', alpha=1)
plt.scatter(3.84, 34.62, marker='*', s=100, color='yellow')

plt.show()